# Disaster XAI-Net — Significance tests for the model comparison

Answers one question with evidence: **are the differences between the ConvNeXt, ViT and fusion models on the held-out test set larger than chance?**

* **Paired bootstrap on the test set.** All models are scored on the *same* resampled test sets, so the comparison is paired. Resampling is done by **source tile** (cluster bootstrap): buildings in one tile share imagery and conditions, so resampling individual buildings would overstate certainty. A building-level bootstrap is reported too, for reference.
* **Fold-wise CV comparison.** Wins/losses across the 5 shared folds (with only 5 folds, no test can reach p < 0.05 two-sided, so wins are reported descriptively).

Inputs (attach as Kaggle inputs, or place next to the notebook):
* the fusion run: `disaster_xai_net_master_results/final_test/stage2_test_predictions.csv`
* the baselines: `convnext_results/final_test/stage2_predictions.csv`, `vit_results/final_test/stage2_predictions.csv`
* optional: `cv_foldwise_all_models.csv`

Nothing here trains or tunes anything; it only reads saved test predictions.

In [ ]:
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_BOOT = 5000
SEED = 0
OUT = os.environ.get("SIG_OUT", "/kaggle/working/significance" if os.path.isdir("/kaggle") else "significance")
os.makedirs(OUT, exist_ok=True)
DAMAGE = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
SEARCH = [p for p in [os.environ.get("SIG_ROOT"), "/kaggle/input", "/kaggle/working", "."] if p and os.path.isdir(p)]

ALL_CSV = sorted({p for root in SEARCH for p in glob.glob(os.path.join(root, "**", "final_test", "stage2*predictions*.csv"), recursive=True)})
print("candidate test-prediction files:\n  " + "\n  ".join(ALL_CSV))

def find(include, exclude=()):
    """first candidate whose path contains any `include` keyword and no `exclude` keyword (case-insensitive)"""
    for p in ALL_CSV:
        lp = p.lower()
        if any(k in lp for k in include) and not any(k in lp for k in exclude):
            return p
    return None

# Optional manual override if the automatic match picks the wrong file:
#   FUSION_CSV = "/kaggle/input/<dataset>/final_test/stage2_test_predictions.csv"
FUSION_CSV = CONVNEXT_CSV = VIT_CSV = None
FILES = {
    'Fusion (ConvNeXt+ViT)': FUSION_CSV or find(['disaster_xai_net_master_results', 'disaster-xai-results-archive', 'fusion', 'ensemble']),
    'ConvNeXt-Tiny':         CONVNEXT_CSV or find(['convnext', 'convexnet'], exclude=['disaster-xai-results-archive', 'master_results', 'fusion', 'ensemble']),
    'ViT-B/16':              VIT_CSV or find(['vit'], exclude=['disaster-xai-results-archive', 'master_results', 'fusion', 'ensemble', 'convnext']),
}
assert len({v for v in FILES.values() if v}) == len([v for v in FILES.values() if v]), \
    "two models matched the same file - set FUSION_CSV / CONVNEXT_CSV / VIT_CSV by hand"
print(json.dumps(FILES, indent=1))
FILES = {k: v for k, v in FILES.items() if v}
assert len(FILES) >= 2, "need at least two models' test prediction files - attach the result folders"

In [ ]:
def load_preds(path):
    df = pd.read_csv(path)
    pcols = [f'prob_{n}' for n in DAMAGE]
    assert all(c in df for c in pcols), f"{path}: missing probability columns"
    y = df['ground_truth'].map({n: i for i, n in enumerate(DAMAGE)})
    assert y.notna().all(), f"{path}: unknown ground-truth labels"
    tile = df['parent_image_id'] if 'parent_image_id' in df else df['crop_id'].astype(str).str.split('#').str[0]
    return pd.DataFrame({'crop_id': df['crop_id'].astype(str), 'tile': tile.astype(str), 'y': y.astype(int),
                         'pred': df[pcols].values.argmax(1),
                         'pred_bin': (df[pcols[1:]].values.sum(1) > 0.5).astype(int)})

P = {m: load_preds(p) for m, p in FILES.items()}
base = None
for m, df in P.items():
    d = df.set_index('crop_id')
    if base is None:
        base = d[['tile', 'y']].copy()
    common = base.index.intersection(d.index)
    assert len(common) == len(base) == len(d), f"{m}: test sets differ ({len(d)} vs {len(base)} crops)"
    assert (d.loc[base.index, 'y'].values == base['y'].values).all(), f"{m}: ground truth differs"
    base[f'pred__{m}'] = d.loc[base.index, 'pred'].values
    base[f'bin__{m}'] = d.loc[base.index, 'pred_bin'].values
MODELS = list(P)
tiles = base['tile'].astype('category')
tile_idx = tiles.cat.codes.values
n_tiles = len(tiles.cat.categories)
print(f"{len(base)} test buildings in {n_tiles} tiles; models: {MODELS}")

In [ ]:
def tile_cms(y, pred, k):
    """per-tile confusion matrices, shape (n_tiles, k, k)"""
    cm = np.zeros((n_tiles, k, k), np.int64)
    np.add.at(cm, (tile_idx, y, pred), 1)
    return cm

def metrics_from_cm(cm):
    """cm: (..., k, k) counts -> dict of arrays (macro-F1, QWK, MCC, accuracy)"""
    cm = cm.astype(float)
    k = cm.shape[-1]
    tp = np.diagonal(cm, axis1=-2, axis2=-1)
    row, col, n = cm.sum(-1), cm.sum(-2), cm.sum((-1, -2))
    with np.errstate(invalid='ignore', divide='ignore'):
        f1 = np.where(row + col > 0, 2 * tp / (row + col), 0.0)
        macro_f1 = f1.mean(-1)
        acc = tp.sum(-1) / n
        cov_yp = n * tp.sum(-1) - (row * col).sum(-1)
        mcc = cov_yp / np.sqrt((n ** 2 - (row ** 2).sum(-1)) * (n ** 2 - (col ** 2).sum(-1)))
        i, j = np.meshgrid(np.arange(k), np.arange(k), indexing='ij')
        W = (i - j) ** 2 / (k - 1) ** 2
        E = row[..., :, None] * col[..., None, :] / n[..., None, None]
        qwk = 1 - (W * cm).sum((-1, -2)) / (W * E).sum((-1, -2))
    return {'macro_f1': macro_f1, 'qwk': qwk, 'mcc': mcc, 'accuracy': acc}

y = base['y'].values
CMS = {('multiclass', m): tile_cms(y, base[f'pred__{m}'].values, 4) for m in MODELS}
CMS.update({('binary', m): tile_cms((y > 0).astype(int), base[f'bin__{m}'].values, 2) for m in MODELS})

rng = np.random.default_rng(SEED)
# cluster bootstrap: each replicate draws n_tiles tiles with replacement -> count of each tile
counts = rng.multinomial(n_tiles, np.full(n_tiles, 1 / n_tiles), size=N_BOOT)   # (B, n_tiles)

def boot(view, m):
    cm = CMS[(view, m)]
    point = metrics_from_cm(cm.sum(0))
    rep = metrics_from_cm(np.einsum('bt,tij->bij', counts, cm))
    return point, rep

VIEWS = {'multiclass': ['macro_f1', 'qwk', 'mcc', 'accuracy'], 'binary': ['macro_f1', 'mcc', 'accuracy']}
POINT, REP = {}, {}
for view in VIEWS:
    for m in MODELS:
        POINT[(view, m)], REP[(view, m)] = boot(view, m)

In [ ]:
rows = []
for view, mets in VIEWS.items():
    for m in MODELS:
        for k in mets:
            r = REP[(view, m)][k]
            rows.append({'view': view, 'model': m, 'metric': k, 'value': POINT[(view, m)][k],
                         'ci_low': np.nanpercentile(r, 2.5), 'ci_high': np.nanpercentile(r, 97.5)})
per_model = pd.DataFrame(rows)
per_model.to_csv(os.path.join(OUT, "test_metrics_with_95ci.csv"), index=False)

pairs = [(a, b) for i, a in enumerate(MODELS) for b in MODELS[i + 1:]]
rows = []
for view, mets in VIEWS.items():
    for a, b in pairs:
        for k in mets:
            d = REP[(view, a)][k] - REP[(view, b)][k]
            d = d[~np.isnan(d)]
            p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
            rows.append({'view': view, 'A': a, 'B': b, 'metric': k,
                         'A_value': POINT[(view, a)][k], 'B_value': POINT[(view, b)][k],
                         'diff_A_minus_B': POINT[(view, a)][k] - POINT[(view, b)][k],
                         'ci_low': np.percentile(d, 2.5), 'ci_high': np.percentile(d, 97.5), 'p_value': p,
                         'significant_at_0.05': bool(p < 0.05)})
pairwise = pd.DataFrame(rows)
pairwise.to_csv(os.path.join(OUT, "pairwise_differences_tile_bootstrap.csv"), index=False)
pd.set_option('display.width', 220)
print(f"PAIRED CLUSTER BOOTSTRAP (by tile, {N_BOOT} replicates) - held-out test, natural distribution\n")
print(pairwise.round(4).to_string(index=False))

# building-level bootstrap for reference (ignores within-tile correlation -> narrower intervals)
cnt_b = rng.multinomial(len(base), np.full(len(base), 1 / len(base)), size=min(N_BOOT, 2000))
ref = []
for a, b in pairs:
    def rep_b(m):
        cm = np.zeros((len(base), 4, 4)); cm[np.arange(len(base)), y, base[f'pred__{m}'].values] = 1
        return metrics_from_cm(np.einsum('bn,nij->bij', cnt_b, cm))['macro_f1']
    d = rep_b(a) - rep_b(b)
    ref.append({'A': a, 'B': b, 'metric': 'macro_f1 (multiclass)', 'ci_low': np.percentile(d, 2.5),
                'ci_high': np.percentile(d, 97.5), 'p_value': min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))})
ref = pd.DataFrame(ref)
ref.to_csv(os.path.join(OUT, "pairwise_macro_f1_building_bootstrap_reference.csv"), index=False)
print("\nreference: building-level bootstrap (too optimistic - report the tile-level one)\n" + ref.round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 0.9 + 0.55 * len(pairs)), sharey=True)
for ax, k in zip(axes, ['macro_f1', 'qwk', 'mcc']):
    sub = pairwise[(pairwise['view'] == 'multiclass') & (pairwise['metric'] == k)].reset_index(drop=True)
    yy = np.arange(len(sub))
    ax.errorbar(sub['diff_A_minus_B'], yy, xerr=[sub['diff_A_minus_B'] - sub['ci_low'], sub['ci_high'] - sub['diff_A_minus_B']],
                fmt='o', capsize=4)
    ax.axvline(0, color='k', lw=1, ls='--')
    ax.set_yticks(yy); ax.set_yticklabels([f"{a}\nminus {b}" for a, b in zip(sub['A'], sub['B'])], fontsize=8)
    ax.set_title(f"test {k}: difference, 95% CI"); ax.grid(alpha=.3, axis='x')
plt.tight_layout(); plt.savefig(os.path.join(OUT, "pairwise_differences_forest.png"), dpi=150, bbox_inches='tight')
plt.show()

### Fold-wise CV comparison (descriptive)

In [ ]:
cvf = find("cv_foldwise_all_models.csv", "comparison/cv_foldwise_all_models.csv")
if cvf:
    cv = pd.read_csv(cvf)
    cv = cv[cv['view'] == 'stage2_multiclass']
    short = {m: ('Fusion' if 'fusion' in m.lower() else 'ConvNeXt' if 'convnext' in m.lower() else 'ViT' if 'vit' in m.lower() else m)
             for m in cv['model'].unique()}
    cv['m'] = cv['model'].map(short)
    fcols = [c for c in cv.columns if c.startswith('fold_')]
    rows = []
    for metric in ['f1_macro', 'qwk', 'mcc']:
        t = cv[cv['metric'] == metric].set_index('m')[fcols]
        for a in t.index:
            for b in t.index:
                if a >= b: continue
                d = t.loc[a].values - t.loc[b].values
                rows.append({'metric': metric, 'A': a, 'B': b, 'A_wins': int((d > 0).sum()), 'B_wins': int((d < 0).sum()),
                             'mean_diff': d.mean(), 'min_diff': d.min(), 'max_diff': d.max()})
    cvcmp = pd.DataFrame(rows)
    cvcmp.to_csv(os.path.join(OUT, "cv_foldwise_wins.csv"), index=False)
    print(cvcmp.round(4).to_string(index=False))
    print("\nwith 5 folds the smallest possible two-sided sign-test p-value is 0.0625 - report wins descriptively")
else:
    print("cv_foldwise_all_models.csv not found - skipped")

### How to write it up

Report each model's test value with its tile-bootstrap 95% CI (`test_metrics_with_95ci.csv`) and the paired differences (`pairwise_differences_tile_bootstrap.csv`). If a difference's interval contains 0, write that the models are **statistically indistinguishable** on that metric (paired cluster bootstrap over test tiles, 5,000 replicates, 95% CI). Report the fold-wise wins as supporting, descriptive evidence.